# Формирование сабмита

## Формат:
```csv
event_id,predict
125854726334416,-0.338988
...
```
- `predict` — непрерывное число (score/логит), **не** 0/1
- Нужно покрыть **все 633,683** event_id из sample_submit.csv
- Метрика считается по `average_precision_score` → чем выше predict у фрода, тем лучше

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier
from pathlib import Path
from datetime import datetime
import json

ROOT = Path('/home/vadim/PyPr/hak')
DATA = ROOT / 'main_data'
FEATURES_OUT = ROOT / 'features'
MODELS_OUT = ROOT / 'models'
SUBMIT_OUT = ROOT / 'submissions'
SUBMIT_OUT.mkdir(exist_ok=True)

FEATURE_COLS = [
    'hour', 'weekday', 'month', 'is_night',
    'log_amount', 'operaton_amt', 'is_null_amount',
    'phone_voip_call_state', 'web_rdp_connection', 'compromised',
    'developer_tools', 'security_flags_sum',
    'event_type_nm', 'is_high_risk_type',
    'mcc_code', 'is_null_mcc',
    'channel_indicator_type', 'channel_indicator_sub_type', 'currency_iso_cd',
    'battery', 'operating_system_type', 'pos_cd',
    'cnt_1h', 'cnt_6h', 'cnt_24h', 'cnt_7d', 'cnt_30d',
    'amt_sum_1h', 'amt_sum_6h', 'amt_sum_24h', 'amt_sum_7d', 'amt_sum_30d',
    'secs_since_last', 'voip_cnt_24h',
    'is_new_mcc_code', 'is_new_channel_indicator_type', 'is_new_currency_iso_cd',
    'cum_unique_mcc_approx',
    'session_ops_before', 'session_amt_before',
    'timezone',
]

print('Ready')

## 1. Загружаем модели

In [ ]:
lgbm_model = lgb.Booster(model_file=str(MODELS_OUT / 'lgbm_final.txt'))
cat_model = CatBoostClassifier()
cat_model.load_model(str(MODELS_OUT / 'catboost.cbm'))

with open(MODELS_OUT / 'weights.json') as f:
    weights = json.load(f)

print('Models loaded')
print(f'Weights — lgbm: {weights["lgbm"]:.3f}, catboost: {weights["catboost"]:.3f}')

## 2. Загружаем test фичи

In [ ]:
print('Loading test features...')
df_test = pl.read_parquet(FEATURES_OUT / 'test_features.parquet')
print(f'Test shape: {df_test.shape}')

X_test = df_test.select(FEATURE_COLS).to_pandas()
event_ids = df_test['event_id'].to_numpy()

print(f'X_test shape: {X_test.shape}')

## 3. Предсказания и ансамбль

In [ ]:
print('Predicting LightGBM...')
lgbm_preds = lgbm_model.predict(X_test)

print('Predicting CatBoost...')
cat_preds = cat_model.predict_proba(X_test)[:, 1]

ensemble_preds = lgbm_preds * weights['lgbm'] + cat_preds * weights['catboost']

print(f'Predictions: min={ensemble_preds.min():.4f}, max={ensemble_preds.max():.4f}, mean={ensemble_preds.mean():.4f}')

## 4. Проверяем покрытие и сохраняем

In [ ]:
sample = pl.read_csv(DATA / 'sample_submit.csv')
print(f'Sample: {len(sample):,} rows')

# Создаём наш сабмит
submit = pl.DataFrame({
    'event_id': event_ids,
    'predict': ensemble_preds,
})

# Выровниваем по порядку sample_submit
submit = sample.select('event_id').join(submit, on='event_id', how='left')

# Проверяем nulls
n_null = submit['predict'].is_null().sum()
print(f'Missing predictions: {n_null}')
if n_null > 0:
    # Заполняем median для missing
    median_pred = submit['predict'].drop_nulls().median()
    submit = submit.with_columns(pl.col('predict').fill_null(median_pred))
    print(f'Filled {n_null} missing with median={median_pred:.4f}')

# Сохраняем
ts = datetime.now().strftime('%Y%m%d_%H%M')
out_path = SUBMIT_OUT / f'submit_{ts}.csv'
submit.write_csv(out_path)
print(f'\nSaved: {out_path}')
print(f'Rows: {len(submit):,}')
print(submit.head(5))